# 🧮 AIMO 3 - Llama-3.3-Nemotron-Super-49B-v1.5 Inference

This notebook implements inference using NVIDIA's **Llama-3.3-Nemotron-Super-49B-v1.5** for the AI Mathematical Olympiad Progress Prize 3.

## Model Highlights
- **Architecture**: 49B parameters via Neural Architecture Search (NAS) from Llama-3.3-70B
- **Optimized for**: Single H100-80GB GPU with FP8 precision (~50GB VRAM)
- **Reasoning Mode**: Built-in `<think>` tags for chain-of-thought reasoning
- **Context Length**: 128K tokens
- **Benchmark Performance**:
  - MATH500: 97.4% pass@1
  - AIME 2024: 87.5% pass@1
  - AIME 2025: 82.71% pass@1

Reference: [nvidia/Llama-3_3-Nemotron-Super-49B-v1_5](https://huggingface.co/nvidia/Llama-3_3-Nemotron-Super-49B-v1_5)


In [1]:
# Uninstall conflicting packages (Kaggle environment)
%pip uninstall --yes "tensorflow" "matplotlib" "keras" "scikit-learn" -q 2>/dev/null || true


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Verify GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    assert torch.cuda.device_count() >= 1, "GPU not enabled"


PyTorch version: 2.9.1+cu130
CUDA available: True
GPU: NVIDIA H100 PCIe
VRAM: 85.0 GB


In [3]:
# Environment setup
import os
import sys
import ctypes

# Set CUDA 12 library paths for vLLM (required when system has CUDA 13.x)
# This must be done BEFORE importing vLLM - both LD_LIBRARY_PATH (for subprocesses) and ctypes (for main process)
nvidia_pkg_base = os.path.expanduser("~/.conda/envs/aimo/lib/python3.12/site-packages/nvidia")
cuda12_lib_paths = [
    f"{nvidia_pkg_base}/cuda_runtime/lib",
    f"{nvidia_pkg_base}/cublas/lib",
    f"{nvidia_pkg_base}/cuda_nvrtc/lib",
    f"{nvidia_pkg_base}/nvjitlink/lib",
    f"{nvidia_pkg_base}/cufft/lib",
    f"{nvidia_pkg_base}/cusparse/lib",
    f"{nvidia_pkg_base}/cusolver/lib",
]

# Set LD_LIBRARY_PATH so subprocesses can find CUDA 12 libs
existing_ld_path = os.environ.get("LD_LIBRARY_PATH", "")
new_paths = [p for p in cuda12_lib_paths if os.path.exists(p) and p not in existing_ld_path]
if new_paths:
    os.environ["LD_LIBRARY_PATH"] = ":".join(new_paths) + ":" + existing_ld_path
    print(f"🔧 Set LD_LIBRARY_PATH with {len(new_paths)} CUDA 12 library paths")

# Pre-load key libraries into current process
cuda12_libs = [
    f"{nvidia_pkg_base}/cuda_runtime/lib/libcudart.so.12",
    f"{nvidia_pkg_base}/cublas/lib/libcublas.so.12",
    f"{nvidia_pkg_base}/cublas/lib/libcublasLt.so.12",
    f"{nvidia_pkg_base}/cuda_nvrtc/lib/libnvrtc.so.12",
]

print("🔧 Pre-loading CUDA 12 libraries for vLLM compatibility...")
for lib_path in cuda12_libs:
    if os.path.exists(lib_path):
        try:
            ctypes.CDLL(lib_path, mode=ctypes.RTLD_GLOBAL)
            print(f"   ✅ {os.path.basename(lib_path)}")
        except OSError as e:
            print(f"   ⚠️  {os.path.basename(lib_path)}: {e}")
    else:
        print(f"   ⚠️  Not found: {os.path.basename(lib_path)}")

os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Single GPU
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Add kaggle_evaluation to path (for local development)
# On Kaggle, this module is pre-installed
kaggle_eval_path = os.path.expanduser("~/AIMOPP3/datasets/aimo3")
if os.path.exists(kaggle_eval_path) and kaggle_eval_path not in sys.path:
    sys.path.insert(0, kaggle_eval_path)
    print(f"✅ Added kaggle_evaluation path: {kaggle_eval_path}")


🔧 Set LD_LIBRARY_PATH with 7 CUDA 12 library paths
🔧 Pre-loading CUDA 12 libraries for vLLM compatibility...
   ✅ libcudart.so.12
   ✅ libcublas.so.12
   ✅ libcublasLt.so.12
   ✅ libnvrtc.so.12
✅ Added kaggle_evaluation path: /localhome/local-tranminhq/AIMOPP3/datasets/aimo3


In [4]:
# Core imports
import time
import warnings
import re
import tempfile
import subprocess
from collections import Counter, defaultdict
from typing import Optional

import numpy as np
import pandas as pd
import polars as pl

from transformers import set_seed
from vllm import LLM, SamplingParams
import kaggle_evaluation.aimo_3_inference_server

# Set seed for reproducibility
set_seed(42)
pd.set_option('display.max_colwidth', None)
warnings.simplefilter('ignore')

# Competition time limit: 4 hours 45 minutes
cutoff_time = time.time() + (4 * 60 + 45) * 60
print(f"⏱️  Cutoff time set: {(4*60+45)} minutes from now")


⏱️  Cutoff time set: 285 minutes from now


## 🔧 Model Configuration

The Nemotron-Super-49B-v1.5 has a special **Reasoning Mode**:
- **Reasoning ON (default)**: Model generates internal reasoning with `<think>` tags
- **Reasoning OFF**: Add `/no_think` to system prompt for faster, direct responses

For AIMO, we use **Reasoning ON** with recommended settings:
- Temperature: 0.6
- Top-P: 0.95


In [5]:
# Model path configuration
# Option 1: Kaggle Input (for competition)
# LLM_MODEL_PATH = '/kaggle/input/nemotron-super-49b/transformers/v1_5/1'

# Option 2: Local checkpoint
LLM_MODEL_PATH = '/localhome/local-tranminhq/AIMOPP3/checkpoints/nemotron-super-49b-v1_5'

# Option 3: HuggingFace Hub (requires internet)
# LLM_MODEL_PATH = 'nvidia/Llama-3_3-Nemotron-Super-49B-v1_5'

print(f"📁 Model path: {LLM_MODEL_PATH}")


📁 Model path: /localhome/local-tranminhq/AIMOPP3/checkpoints/nemotron-super-49b-v1_5


In [6]:
# Initialize the model with vLLM (Single H100 with 4-bit quantization)
print("🚀 Loading Llama-3.3-Nemotron-Super-49B-v1.5 with 4-bit quantization...")
print("   This may take 2-5 minutes...")

llm = LLM(
    LLM_MODEL_PATH,
    dtype="bfloat16",              # Compute dtype
    quantization="bitsandbytes",   # 4-bit NF4 quantization (~25GB weights)
    load_format="bitsandbytes",    # Load directly in quantized format
    max_num_seqs=1,                # Batch size
    max_model_len=131072,          # 128K context length
    trust_remote_code=True,        # Required for custom architecture
    tensor_parallel_size=1,        # Single GPU
    gpu_memory_utilization=0.90,   # Good utilization
    enforce_eager=True,            # Disable CUDA graphs for compatibility
)

print("✅ Model loaded successfully with 4-bit quantization!")


🚀 Loading Llama-3.3-Nemotron-Super-49B-v1.5 with 4-bit quantization...
   This may take 2-5 minutes...
INFO 12-28 18:30:06 [utils.py:253] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'dtype': 'bfloat16', 'max_model_len': 131072, 'max_num_seqs': 1, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'enforce_eager': True, 'model': '/localhome/local-tranminhq/AIMOPP3/checkpoints/nemotron-super-49b-v1_5'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 12-28 18:30:09 [model.py:514] Resolved architecture: DeciLMForCausalLM
INFO 12-28 18:30:09 [model.py:1661] Using max model len 131072
INFO 12-28 18:30:09 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=16384.
WARNING 12-28 18:30:11 [vllm.py:622] Enforce eager set, overriding optimization level to -O0
INFO 12-28 18:30:11 [vllm.py:722] Cudagraph is disabled under eager mode
WARNING 12-28 18:30:12 [system_utils.py:136] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore_DP0 pid=1005864) INFO 12-28 18:30:23 [core.py:93] Initializing a V1 LLM engine (v0.13.0) with config: model='/localhome/local-tranminhq/AIMOPP3/checkpoints/nemotron-super-49b-v1_5', speculative_config=None, tokenizer='/localhome/local-tranminhq/AIMOPP3/checkpoints/nemotron-super-49b-v1_5', skip

Loading safetensors checkpoint shards:   0% Completed | 0/21 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:   5% Completed | 1/21 [00:00<00:14,  1.42it/s]
Loading safetensors checkpoint shards:  10% Completed | 2/21 [00:01<00:14,  1.30it/s]
Loading safetensors checkpoint shards:  14% Completed | 3/21 [00:02<00:12,  1.40it/s]
Loading safetensors checkpoint shards:  19% Completed | 4/21 [00:03<00:13,  1.24it/s]
Loading safetensors checkpoint shards:  24% Completed | 5/21 [00:04<00:13,  1.19it/s]
Loading safetensors checkpoint shards:  29% Completed | 6/21 [00:04<00:12,  1.21it/s]
Loading safetensors checkpoint shards:  33% Completed | 7/21 [00:05<00:11,  1.21it/s]
Loading safetensors checkpoint shards:  38% Completed | 8/21 [00:06<00:10,  1.20it/s]
Loading safetensors checkpoint shards:  43% Completed | 9/21 [00:07<00:09,  1.20it/s]
Loading safetensors checkpoint shards:  48% Completed | 10/21 [00:08<00:09,  1.20it/s]
Loading safetensors checkpoint shards:  52% Completed | 11/21

(EngineCore_DP0 pid=1005864) INFO 12-28 18:30:44 [gpu_model_runner.py:3659] Model loading took 28.9871 GiB memory and 18.864709 seconds
(EngineCore_DP0 pid=1005864) INFO 12-28 18:30:49 [gpu_worker.py:375] Available KV cache memory: 38.30 GiB
(EngineCore_DP0 pid=1005864) INFO 12-28 18:30:50 [kv_cache_utils.py:1291] GPU KV cache size: 204,880 tokens
(EngineCore_DP0 pid=1005864) INFO 12-28 18:30:50 [kv_cache_utils.py:1296] Maximum concurrency for 131,072 tokens per request: 1.56x


(EngineCore_DP0 pid=1005864) 2025-12-28 18:30:50,175 - INFO - autotuner.py:256 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore_DP0 pid=1005864) 2025-12-28 18:30:52,495 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process ends


(EngineCore_DP0 pid=1005864) INFO 12-28 18:30:54 [core.py:259] init engine (profile, create kv cache, warmup model) took 10.15 seconds
(EngineCore_DP0 pid=1005864) WARNING 12-28 18:30:55 [vllm.py:629] Inductor compilation was disabled by user settings,Optimizations settings that are only active duringInductor compilation will be ignored.
(EngineCore_DP0 pid=1005864) INFO 12-28 18:30:55 [vllm.py:722] Cudagraph is disabled under eager mode
INFO 12-28 18:30:55 [llm.py:360] Supported tasks: ['generate']
✅ Model loaded successfully with 4-bit quantization!


In [7]:
tokenizer = llm.get_tokenizer()
print(f"📝 Tokenizer loaded: {tokenizer.name_or_path}")


📝 Tokenizer loaded: /localhome/local-tranminhq/AIMOPP3/checkpoints/nemotron-super-49b-v1_5


In [8]:
# Sampling parameters optimized for Nemotron Reasoning Mode
# Reference: https://huggingface.co/nvidia/Llama-3_3-Nemotron-Super-49B-v1_5

sampling_params = SamplingParams(
    temperature=0.6,           # Recommended for Reasoning ON mode
    top_p=0.95,                # Recommended for Reasoning ON mode
    min_p=0.01,                # Filter low probability tokens
    skip_special_tokens=True,
    max_tokens=32768,          # Allow long reasoning with 128K context
)

print("⚙️  Sampling params configured:")
print(f"   temperature={sampling_params.temperature}")
print(f"   top_p={sampling_params.top_p}")
print(f"   max_tokens={sampling_params.max_tokens}")


⚙️  Sampling params configured:
   temperature=0.6
   top_p=0.95
   max_tokens=32768


## 📐 Answer Extraction Utilities


In [9]:
def extract_boxed_answers(text: str) -> list[int]:
    """Extract all \\boxed{} answers from model output."""
    pattern = r'oxed{(.*?)}'
    matches = re.findall(pattern, text)
    if not matches:
        return []
    
    ans = []
    for content in matches:
        if content.isdigit():
            num = content
        else:
            nums = re.findall(r'\d+', content)
            if not nums:
                continue
            num = nums[-1]
        ans.append(int(num))
    return ans


def select_answer(answers: list) -> int:
    """Select final answer using majority voting."""
    valid_answers = []
    for answer in answers:
        try:
            if int(answer) != float(answer):
                continue
            # AIMO 3 rules: answer must be 0-99999
            if 0 <= int(answer) <= 99999:
                valid_answers.append(int(answer))
        except:
            pass
    
    if not valid_answers:
        print("⚠️  No valid answers found, guessing 42")
        return 42
    
    answer, count = Counter(valid_answers).most_common(1)[0]
    print(f"   Selected answer: {answer} (appeared {count} times)")
    return answer


## 🐍 Python Code Execution (Tool Use)


In [10]:
def extract_python_code(text: str) -> list[str]:
    """Extract Python code blocks from model output."""
    pattern = r'```python\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)
    return matches


def process_python_code(query: str) -> str:
    """Add standard imports to Python code."""
    imports = "import math\nimport numpy as np\nimport sympy as sp\nfrom fractions import Fraction\n"
    return imports + query.strip()


class PythonREPL:
    """Safe Python code executor with timeout."""
    
    def __init__(self, timeout: int = 10):
        self.timeout = timeout

    def __call__(self, query: str) -> tuple[bool, str]:
        with tempfile.TemporaryDirectory() as temp_dir:
            temp_file_path = os.path.join(temp_dir, "tmp.py")
            with open(temp_file_path, "w", encoding="utf-8") as f:
                f.write(query)
            
            try:
                result = subprocess.run(
                    ["python3", temp_file_path],
                    capture_output=True,
                    check=False,
                    text=True,
                    timeout=self.timeout,
                )
            except subprocess.TimeoutExpired:
                return False, f"Execution timed out after {self.timeout} seconds."

            stdout = result.stdout.strip()
            stderr = result.stderr.strip()

            if result.returncode == 0:
                return True, stdout
            else:
                error_lines = stderr.split("\n")
                cleaned_errors = [line.replace(temp_file_path, "<code>") for line in error_lines]
                return False, "\n".join(cleaned_errors)


## 🧠 Prompt Engineering for Nemotron Reasoning Mode

Nemotron-Super-49B-v1.5 uses **Reasoning Mode ON by default**. We leverage this with diverse system prompts for self-consistency.


In [11]:
# System prompts for diverse reasoning paths
# Note: Do NOT include /no_think - we want Reasoning Mode ON

SYSTEM_PROMPTS = [
#     # Prompt 1: Standard mathematical reasoning
#     """You are an expert mathematician solving competition problems. 
# Think step-by-step and verify each step before proceeding.
# Put your final answer as a non-negative integer in \\boxed{}.""",
    
#     # Prompt 2: Self-verification focus
#     """You are solving an International Mathematical Olympiad problem.
# After each major step, verify your work before continuing.
# If you find an error, backtrack and try a different approach.
# Final answer must be a non-negative integer (0-99999) in \\boxed{}.""",
    
#     # Prompt 3: Computational approach
#     """You are a computational mathematician. For this problem:
# 1. Identify the mathematical structure
# 2. Consider both analytical and computational approaches
# 3. Verify your answer satisfies all constraints
# Put the final integer answer in \\boxed{}.""",
    
#     # Prompt 4: Problem decomposition
#     """Break this problem into smaller subproblems.
# Solve each subproblem carefully, then combine the results.
# Double-check arithmetic calculations.
# Final answer: non-negative integer in \\boxed{}.""",
    
#     # Prompt 5: Multiple methods
#     """Consider multiple solution approaches for this problem.
# Choose the most elegant method and execute it precisely.
# Verify the answer makes sense given the problem constraints.
# Box your final integer answer: \\boxed{answer}.""",

    # Gemimi prompt
    """You are a computational mathematician. To solve the problem, write a robust Python script that calculates the answer.
Define the variables and constraints clearly.
Implement the logic in a function.
Print the final answer at the end of the script.
Do not guess; calculate.
Final answer must be a non-negative integer (0-99999) in \\boxed{answer}
    """
]

print(f"📋 Configured {len(SYSTEM_PROMPTS)} diverse system prompts for self-consistency")


📋 Configured 1 diverse system prompts for self-consistency


In [12]:
# Load few-shot examples from examples.txt
# We select 3 diverse examples to demonstrate the expected format

EXAMPLES_PATH = "/localhome/local-tranminhq/AIMOPP3/examples.txt"

FEW_SHOT_EXAMPLES = [
    # Example 1: Geometry (AIMO2 - Triangle/Circumradius)
    {
        "problem": "Triangle $ABC$ has side length $AB = 120$ and circumradius $R = 100$. Let $D$ be the foot of the perpendicular from $C$ to the line $AB$. What is the greatest possible length of segment $CD$?",
        "solution": """Let $O$ be the circumcentre of triangle $ABC$. Then
$$\\text{dist}(O, AB) = \\sqrt{R^2 - (AB/2)^2} = \\sqrt{100^2 - 60^2} = 80$$
by Pythagoras.

Since $C$ must be on the circle with centre $O$ and radius $OA$, the largest possible altitude $h_c$ is attained when $C$ is the mid-point of the larger arc $AB$ of the circumcircle (i.e. on the perpendicular bisector of $AB$) in which case we have:
$$h_c = \\text{dist}(O, AB) + R = 80 + 100 = \\boxed{180}.$$"""
    },
    
    # Example 2: Number Theory (AIMO2 - Digit sum)
    {
        "problem": "For a positive integer $n$, let $S(n)$ denote the sum of the digits of $n$ in base 10. Compute $S(S(1)+S(2)+\\cdots+S(N))$ with $N=10^{100}-2$.",
        "solution": """For each integer $k$ in range $0 \\leq k \\leq 10^{100}-1$, we have:
$$k + (10^{100} - k - 1) = 10^{100} - 1$$

This is a string of 100 nines. For each position, the digits of $k$ and $10^{100}-1-k$ sum to 9.

Therefore $S(k) + S(10^{100} - 1 - k) = 9 \\times 100 = 900$.

The sum $S(0) + S(1) + ... + S(10^{100}-1)$ can be computed by pairing:
$$\\sum_{k=0}^{10^{100}-1} S(k) = \\frac{10^{100}}{2} \\times 900 = 450 \\times 10^{100}$$

Now we need $S(1) + ... + S(10^{100}-2) = 450 \\times 10^{100} - S(0) - S(10^{100}-1) = 450 \\times 10^{100} - 0 - 900 = 450 \\times 10^{100} - 900$.

In decimal, this is $449999...9100$ (with 97 nines).

The digit sum is $4 + 4 + 9 \\times 97 + 1 + 0 + 0 = 4 + 4 + 873 + 1 = 882$.

Wait, let me recalculate: $450 \\times 10^{100} - 900 = 4499...9100$ where we have 98 nines.
$S(4499...9100) = 4 + 4 + 9 \\times 98 + 1 = 4 + 4 + 882 + 1 = \\boxed{891}.$"""
    },
    
    # Example 3: Combinatorics (AIMO3 - Delightful sequences)
    {
        "problem": "We call a sequence $a_1, a_2, \\ldots$ of non-negative integers \\textit{delightful} if there exists a positive integer $N$ such that for all $n > N$, $a_n = 0$, and for all $i \\geq 1$, $a_i$ counts the number of multiples of $i$ in $a_1, a_2, \\ldots, a_N$. How many delightful sequences of non-negative integers are there?",
        "solution": """We analyze sequences where $a_i$ counts multiples of $i$ in $a_1, \\ldots, a_N$.

Note $a_1$ counts all terms, so $a_1 = N$.

**Case $N = 1$:** We need $a_1 = 1$ and $a_1$ counts multiples of 1 in $\\{a_1\\}$. So $a_1 = 1$. ✓
Sequence: $(1)$

**Case $N = 2$:** We need $a_1 = 2$, and $a_2$ counts even numbers in $\\{a_1, a_2\\} = \\{2, a_2\\}$.
- If $a_2 = 1$: Check multiples of 2 in $\\{2, 1\\}$ → just $\\{2\\}$, count = 1. ✓
- If $a_2 = 2$: Check multiples of 2 in $\\{2, 2\\}$ → both, count = 2. ✓
Sequences: $(2, 1)$ and $(2, 2)$

**Case $N \\geq 3$:** Through careful analysis, contradictions arise for all attempts.

Total: $\\boxed{3}$ delightful sequences."""
    },
]

# Format examples as conversation for few-shot prompting
def format_few_shot_examples(examples: list[dict], max_examples: int = 2) -> str:
    """Format examples as a string to append to user prompt."""
    formatted = "\\n\\nHere are some solved examples for reference:\\n"
    for i, ex in enumerate(examples[:max_examples], 1):
        formatted += f"\\n---\\n**Example {i}:**\\n"
        formatted += f"Problem: {ex['problem']}\\n\\n"
        formatted += f"Solution:\\n{ex['solution']}\\n"
    formatted += "\\n---\\n\\nNow solve the following problem:\\n"
    return formatted

# Load additional examples from examples.txt
def load_examples_from_file(filepath: str) -> list[dict]:
    """Parse examples from examples.txt file."""
    examples = []
    with open(filepath, 'r') as f:
        content = f.read()
    
    # Split by example markers
    import re
    pattern = r'\*\*Problem:\*\*\n(.*?)\n\n\*\*Solution:\*\*\n(.*?)(?=\n-{40,}|\Z)'
    matches = re.findall(pattern, content, re.DOTALL)
    
    for problem, solution in matches:
        examples.append({
            "problem": problem.strip(),
            "solution": solution.strip()
        })
    return examples

# Try to load from examples.txt, fallback to hardcoded
try:
    ALL_EXAMPLES = load_examples_from_file(EXAMPLES_PATH)
    if len(ALL_EXAMPLES) > len(FEW_SHOT_EXAMPLES):
        FEW_SHOT_EXAMPLES = ALL_EXAMPLES
        print(f"📂 Loaded {len(FEW_SHOT_EXAMPLES)} examples from examples.txt")
except Exception as e:
    print(f"⚠️  Could not load examples.txt: {e}")
    print(f"   Using {len(FEW_SHOT_EXAMPLES)} hardcoded examples")

# Number of few-shot examples to include (with 128K context, we can use all!)
NUM_FEW_SHOT = 10  # Use 10 examples for maximum context

print(f"📚 Total available: {len(FEW_SHOT_EXAMPLES)} examples")
print(f"   Using {NUM_FEW_SHOT} examples per prompt")


📂 Loaded 20 examples from examples.txt
📚 Total available: 20 examples
   Using 10 examples per prompt


## 🔄 Batch Inference Pipeline


In [13]:
MessagesBatch = list[list[dict[str, str]]]

# Global tracking for answer contributions
answer_contributions = defaultdict(list)


def batch_message_generate(msg_batch: MessagesBatch) -> MessagesBatch:
    """Generate responses for a batch of conversations."""
    list_of_texts = [
        tokenizer.apply_chat_template(
            conversation=messages,
            tokenize=False,
            add_generation_prompt=True
        )
        for messages in msg_batch
    ]
    
    request_output = llm.generate(
        prompts=list_of_texts,
        sampling_params=sampling_params,
    )
    
    for messages, single_request_output in zip(msg_batch, request_output):
        response_text = single_request_output.outputs[0].text
        messages.append({'role': 'assistant', 'content': response_text})
        
        # Print truncated response for monitoring
        preview = response_text[:500] + "..." if len(response_text) > 500 else response_text
        print(f"   📝 Response preview: {preview[:200]}...")

    return msg_batch


def batch_message_filter(
    msg_batch: MessagesBatch, 
    list_of_idx: list[int]
) -> tuple[MessagesBatch, list[int], list[int]]:
    """Filter messages that have produced answers."""
    global answer_contributions
    
    extracted_answers: list[int] = []
    msgs_to_keep: MessagesBatch = []
    idx_to_keep: list[int] = []
    
    for idx, messages in zip(list_of_idx, msg_batch):
        answers = extract_boxed_answers(messages[-1]['content'])
        
        if answers:
            extracted_answers.extend(answers)
            for answer in answers:
                answer_contributions[answer].append(idx)
        else:
            msgs_to_keep.append(messages)
            idx_to_keep.append(idx)
    
    return msgs_to_keep, extracted_answers, idx_to_keep


def batch_execute_and_get_answer(list_of_messages: MessagesBatch) -> list[int]:
    """Execute Python code from responses and extract answers."""
    ans = []
    repl = PythonREPL(timeout=10)
    
    for messages in list_of_messages:
        python_code_list = extract_python_code(messages[-1]['content'])
        
        for python_code in python_code_list:
            python_code = process_python_code(python_code)
            try:
                success, output = repl(python_code)
                if not success:
                    continue
                
                matches = re.findall(r'(\d+)', output)
                for match in matches:
                    ans.append(int(match))
                    
                print(f"   🐍 Code output: {output[:100]}...")
            except Exception as e:
                print(f"   ⚠️  Code error: {e}")
    
    return ans


## 🎯 Main Prediction Function


In [14]:
def predict_for_question(question: str, max_rounds: int = 1, use_few_shot: bool = True) -> int:
    """Generate prediction for a single question using self-consistency.
    
    Args:
        question: The math problem to solve
        max_rounds: Maximum rounds of generation
        use_few_shot: Whether to include few-shot examples in the prompt
    """
    global answer_contributions
    answer_contributions = defaultdict(list)  # Reset for each question
    
    # Check time limit
    if time.time() > cutoff_time:
        print("⏰ Time limit exceeded, returning default answer")
        return 42
    
    # Build user prompt with optional few-shot examples
    if use_few_shot and NUM_FEW_SHOT > 0:
        few_shot_prefix = format_few_shot_examples(FEW_SHOT_EXAMPLES, NUM_FEW_SHOT)
        user_content = few_shot_prefix + question
        print(f"📚 Using {NUM_FEW_SHOT} few-shot examples")
    else:
        user_content = question
    
    # Create diverse prompts
    msgs_batch: MessagesBatch = [
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ]
        for system_prompt in SYSTEM_PROMPTS
    ]
    
    all_extracted_answers = []
    list_of_idx = list(range(len(msgs_batch)))
    
    for round_idx in range(max_rounds):
        print(f"\n🔄 Round {round_idx + 1}/{max_rounds}")
        
        # Generate responses
        msgs_batch = batch_message_generate(msgs_batch)
        
        # Extract Python code answers
        extracted_python_answer = batch_execute_and_get_answer(msgs_batch)
        
        # Extract boxed answers
        msgs_batch, extracted_answers, list_of_idx = batch_message_filter(msgs_batch, list_of_idx)
        
        # Collect all answers
        all_extracted_answers.extend(extracted_python_answer)
        all_extracted_answers.extend(extracted_answers)
        
        print(f"   📦 Boxed answers: {extracted_answers}")
        print(f"   🐍 Python answers: {extracted_python_answer}")
        print(f"   📊 Total answers so far: {len(all_extracted_answers)}")
        
        if not msgs_batch:
            print("   ✅ All prompts produced answers")
            break
    
    # Majority voting
    answer = select_answer(all_extracted_answers)
    print(f"\n🎯 Final answer: {answer}")
    return answer


In [15]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    """Prediction function for Kaggle evaluation API."""
    id_val = id_.item(0)
    question_str = question.item(0)
    
    print("=" * 60)
    print(f"📋 Problem ID: {id_val}")
    print(f"📝 Question: {question_str[:200]}...")
    print("=" * 60)
    
    predicted_answer = predict_for_question(question_str)
    
    print(f"\n✅ Submitted answer for {id_val}: {predicted_answer}")
    print("=" * 60 + "\n")
    
    return pl.DataFrame({'id': id_val, 'answer': predicted_answer})


In [16]:
# Initialize and run the inference server
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Competition mode
    print("🏆 Running in COMPETITION mode")
    inference_server.serve()
else:
    # Local testing mode - use local test file
    LOCAL_TEST_PATH = '/localhome/local-tranminhq/AIMOPP3/datasets/aimo3/test.csv'
    print(f"🧪 Running in LOCAL TEST mode")
    print(f"   Using: {LOCAL_TEST_PATH}")
    inference_server.run_local_gateway((LOCAL_TEST_PATH,))


🧪 Running in LOCAL TEST mode
   Using: /localhome/local-tranminhq/AIMOPP3/datasets/aimo3/test.csv
📋 Problem ID: 000aaa
📝 Question: What is $1-1$?...
📚 Using 10 few-shot examples

🔄 Round 1/1


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

   📝 Response preview: <think>
Okay, let's tackle this problem: "What is 1-1?" Hmm, that seems straightforward. But wait, maybe there's a trick here. Let me think.

First, the question is simply asking for the result of sub...
   📦 Boxed answers: [79, 250, 0, 0]
   🐍 Python answers: []
   📊 Total answers so far: 4
   ✅ All prompts produced answers
   Selected answer: 0 (appeared 2 times)

🎯 Final answer: 0

✅ Submitted answer for 000aaa: 0

📋 Problem ID: 111bbb
📝 Question: What is $0\times10$?...
📚 Using 10 few-shot examples

🔄 Round 1/1


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

   📝 Response preview: <think>
Okay, let's tackle this problem: "What is 0 × 10?" Hmm, seems straightforward, but maybe there's a trick here. Let me think.

First, I know that multiplication is about adding a number a certa...
   📦 Boxed answers: [0, 0]
   🐍 Python answers: []
   📊 Total answers so far: 2
   ✅ All prompts produced answers
   Selected answer: 0 (appeared 2 times)

🎯 Final answer: 0

✅ Submitted answer for 111bbb: 0

📋 Problem ID: 222ccc
📝 Question: Solve $4+x=4$ for $x$....
📚 Using 10 few-shot examples

🔄 Round 1/1


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

   📝 Response preview: <think>
Okay, let's see. The problem is to solve 4 + x = 4 for x. Hmm, that seems straightforward. Let me think.

So, starting with the equation: 4 + x = 4. To solve for x, I need to isolate x. That m...
   📦 Boxed answers: [0, 0]
   🐍 Python answers: []
   📊 Total answers so far: 2
   ✅ All prompts produced answers
   Selected answer: 0 (appeared 2 times)

🎯 Final answer: 0

✅ Submitted answer for 222ccc: 0



## 🧪 Optional: Manual Testing
